# Head Pose Estimation EDA

This notebook explores the AFLW2000 dataset, the pose-angle annotations, the selected MediaPipe FaceMesh landmarks, and the extracted feature CSV used by the model. The reusable pipeline code lives in `src/`; this notebook is intentionally focused on exploration, visualization, and sanity checks.

## Goals

- Confirm that image files and `.mat` pose annotation files are paired correctly.
- Understand the distribution of pitch, yaw, and roll values.
- Inspect representative and extreme pose examples visually.
- Visualize FaceMesh landmarks, selected model features, and pose axes.
- Review the extracted feature CSV for missing values, duplicates, and basic statistics.
- Identify possible data-quality issues before training.

## Imports And Paths

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.extract_features import extract_features
from src.utils import (
    FEATURE_INDICES,
    annotate_features,
    detect_face_landmarks,
    draw_axes,
    init_face_mesh,
    load_image_and_pose,
    load_pose_angles,
    load_rgb_image,
    normalize_landmark_dataframe,
    pair_image_and_mat_paths,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)


In [ ]:
DATASET_DIR = PROJECT_ROOT / "data" / "raw" / "AFLW2000"
PROCESSED_CSV = PROJECT_ROOT / "data" / "processed" / "features_pose_angles.csv"
SAMPLE_INDEX = 312
RANDOM_STATE = 42

image_paths = sorted(DATASET_DIR.glob("*.jpg"))
mat_paths = sorted(DATASET_DIR.glob("*.mat"))
pairs = pair_image_and_mat_paths(image_paths, mat_paths)

print(f"Dataset directory: {DATASET_DIR.resolve()}")
print(f"Image files: {len(image_paths):,}")
print(f"MAT files: {len(mat_paths):,}")
print(f"Matched image/MAT pairs: {len(pairs):,}")


## Dataset Inventory

The dataset should contain one `.jpg` image and one `.mat` annotation file per sample. Before looking at model features, we first check whether every image has a matching pose file and whether any annotation files are orphaned.

In [ ]:
image_stems = {path.stem for path in image_paths}
mat_stems = {path.stem for path in mat_paths}

missing_mat = sorted(image_stems - mat_stems)
missing_image = sorted(mat_stems - image_stems)

inventory_df = pd.DataFrame(
    {
        "item": ["images", "mat_files", "matched_pairs", "images_without_mat", "mat_without_image"],
        "count": [len(image_paths), len(mat_paths), len(pairs), len(missing_mat), len(missing_image)],
    }
)
display(inventory_df)

print("First missing MAT examples:", missing_mat[:10])
print("First missing image examples:", missing_image[:10])


## Build A Pose-Angle Summary Table

The AFLW2000 `.mat` files store pose parameters. The first three values are `pitch`, `yaw`, and `roll`, which are the labels used for training.

In [ ]:
pose_rows = []
for image_path, mat_path in pairs:
    pitch, yaw, roll = load_pose_angles(mat_path)
    pose_rows.append(
        {
            "stem": image_path.stem,
            "image_path": image_path,
            "mat_path": mat_path,
            "pitch": pitch,
            "yaw": yaw,
            "roll": roll,
        }
    )

pose_df = pd.DataFrame(pose_rows)
pose_df.head()


In [ ]:
pose_df[["pitch", "yaw", "roll"]].describe().T


## Pose-Angle Distributions

These plots show whether the labels are balanced or concentrated around frontal head poses. Strong imbalance can make the model perform better near common angles and worse on rare extreme poses.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, angle in zip(axes, ["pitch", "yaw", "roll"]):
    sns.histplot(pose_df[angle], bins=40, kde=True, ax=ax, color="#377eb8")
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{angle.title()} Distribution")
    ax.set_xlabel(f"{angle} radians")
plt.tight_layout()


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=pose_df[["pitch", "yaw", "roll"]], orient="h", palette="Set2")
plt.title("Pose Angle Spread")
plt.xlabel("Radians")
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(pose_df[["pitch", "yaw", "roll"]].corr(), annot=True, cmap="vlag", center=0)
plt.title("Pose Angle Correlation")
plt.show()


In [ ]:
sns.pairplot(
    pose_df[["pitch", "yaw", "roll"]].sample(min(1000, len(pose_df)), random_state=RANDOM_STATE),
    diag_kind="kde",
    corner=True,
)
plt.suptitle("Pairwise Pose-Angle Relationships", y=1.02)
plt.show()


## Random Sample Grid

A random grid is useful for quickly checking image quality, pose variety, and whether the annotations appear plausible.

In [ ]:
sample_df = pose_df.sample(min(12, len(pose_df)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, (_, row) in zip(axes.ravel(), sample_df.iterrows()):
    img, _, _ = load_rgb_image(row["image_path"])
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"{row['stem']}\nP {row['pitch']:.2f} | Y {row['yaw']:.2f} | R {row['roll']:.2f}",
        fontsize=9,
    )
for ax in axes.ravel()[len(sample_df):]:
    ax.axis("off")
plt.tight_layout()


## Extreme Pose Examples

The model needs to handle rare high-magnitude angles. These examples inspect the largest absolute values for each target.

In [ ]:
extreme_rows = []
for angle in ["pitch", "yaw", "roll"]:
    idx = pose_df[angle].abs().idxmax()
    row = pose_df.loc[idx].copy()
    row["extreme_angle"] = angle
    extreme_rows.append(row)
extreme_df = pd.DataFrame(extreme_rows)
display(extreme_df[["stem", "extreme_angle", "pitch", "yaw", "roll"]])

fig, axes = plt.subplots(1, len(extreme_df), figsize=(15, 5))
for ax, (_, row) in zip(axes, extreme_df.iterrows()):
    img, _, _ = load_rgb_image(row["image_path"])
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"Max abs {row['extreme_angle']}\nP {row['pitch']:.2f} | Y {row['yaw']:.2f} | R {row['roll']:.2f}",
        fontsize=10,
    )
plt.tight_layout()


## FaceMesh And Pose-Axis Inspection

The next cells visualize one sample through the same concepts used in the original notebooks: original image, detected FaceMesh, and 3D pose axes drawn at the nose.

In [ ]:
img, pitch, yaw, roll = load_image_and_pose(SAMPLE_INDEX, DATASET_DIR)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title(f"Original Image - Sample {SAMPLE_INDEX}")
plt.show()


In [ ]:
nose_x, nose_y, annotated_img = detect_face_landmarks(img)

plt.figure(figsize=(10, 8))
plt.imshow(annotated_img)
plt.axis("off")
plt.title("Detected FaceMesh Landmarks")
plt.show()

print(f"Nose position: x={nose_x}, y={nose_y}")


In [ ]:
img_with_axes = draw_axes(img, pitch, yaw, roll, nose_x, nose_y, size=50)

plt.figure(figsize=(10, 8))
plt.imshow(img_with_axes)
plt.axis("off")
plt.title(f"Pose Axes - pitch={pitch:.2f}, yaw={yaw:.2f}, roll={roll:.2f}")
plt.show()


## Selected Feature Points

The model does not use every FaceMesh point. It uses the selected landmarks below. This keeps the feature vector small and interpretable.

In [ ]:
feature_indices_df = pd.DataFrame(
    [{"feature": name.lower(), "mediapipe_index": index} for name, index in FEATURE_INDICES.items()]
)
display(feature_indices_df)


In [ ]:
sample_path = DATASET_DIR / "image00515.jpg"
sample_img, sample_w, sample_h = load_rgb_image(sample_path)
with init_face_mesh() as face_mesh:
    features = extract_features(sample_img, face_mesh)

feature_img = annotate_features(sample_img, features, sample_w, sample_h)

plt.figure(figsize=(10, 8))
plt.imshow(feature_img)
plt.axis("off")
plt.title("Selected Model Feature Points")
plt.show()


## Processed Feature CSV Checks

After running feature extraction, the processed CSV should live at `data/processed/features_pose_angles.csv`. The checks below inspect its shape, missingness, duplicate rows, and numeric summaries.

In [ ]:
if PROCESSED_CSV.exists():
    features_df = pd.read_csv(PROCESSED_CSV)
    print(f"Processed CSV: {PROCESSED_CSV.resolve()}")
    print(f"Shape: {features_df.shape}")
    display(features_df.head())
else:
    features_df = pd.DataFrame()
    print(f"Processed feature CSV not found: {PROCESSED_CSV}")


In [ ]:
if not features_df.empty:
    quality_df = pd.DataFrame(
        {
            "metric": ["rows", "columns", "duplicate_rows", "total_missing_values"],
            "value": [
                len(features_df),
                features_df.shape[1],
                features_df.duplicated().sum(),
                features_df.isna().sum().sum(),
            ],
        }
    )
    display(quality_df)
    display(features_df.describe(include="all").T)


In [ ]:
if not features_df.empty:
    missing_by_column = features_df.isna().sum().sort_values(ascending=False)
    display(missing_by_column.to_frame("missing_count").head(20))

    plt.figure(figsize=(12, 4))
    sns.barplot(x=missing_by_column.index, y=missing_by_column.values, color="#e41a1c")
    plt.xticks(rotation=90)
    plt.title("Missing Values By Feature Column")
    plt.ylabel("Missing count")
    plt.xlabel("Feature")
    plt.tight_layout()
    plt.show()


## Raw Feature Coordinate Distributions

Landmark coordinates are normalized MediaPipe values before the training normalization step. Their distributions should generally stay within the image-normalized range and should not contain unexpected spikes caused by extraction failures.

In [ ]:
if not features_df.empty:
    feature_columns = [col for col in features_df.columns if col not in ["pitch", "yaw", "roll"]]
    melted_features = features_df[feature_columns].melt(var_name="feature", value_name="value")

    plt.figure(figsize=(16, 6))
    sns.boxplot(data=melted_features, x="feature", y="value", color="#4daf4a")
    plt.xticks(rotation=90)
    plt.title("Raw Extracted Landmark Coordinate Distributions")
    plt.xlabel("Feature")
    plt.ylabel("Coordinate value")
    plt.tight_layout()
    plt.show()


## Training-Normalized Feature Checks

Training centers all coordinates around the nose and scales them by the mouth-right to left-eye distance for each dimension. This section shows how that transformation changes feature distributions.

In [ ]:
if not features_df.empty:
    clean_features_df = features_df.dropna().drop_duplicates()
    normalized_features_df = normalize_landmark_dataframe(clean_features_df)
    display(normalized_features_df.head())

    normalized_feature_columns = [
        col for col in normalized_features_df.columns if col not in ["pitch", "yaw", "roll"]
    ]
    normalized_melted = normalized_features_df[normalized_feature_columns].melt(
        var_name="feature",
        value_name="value",
    )

    plt.figure(figsize=(16, 6))
    sns.boxplot(data=normalized_melted, x="feature", y="value", color="#984ea3")
    plt.xticks(rotation=90)
    plt.title("Normalized Landmark Coordinate Distributions")
    plt.xlabel("Feature")
    plt.ylabel("Normalized value")
    plt.tight_layout()
    plt.show()


In [ ]:
if not features_df.empty:
    plt.figure(figsize=(12, 10))
    corr_cols = normalized_feature_columns + ["pitch", "yaw", "roll"]
    sns.heatmap(normalized_features_df[corr_cols].corr(), cmap="vlag", center=0)
    plt.title("Feature And Pose Correlation Heatmap")
    plt.tight_layout()
    plt.show()


## Feature Relationships With Pose Targets

These quick scatter plots help reveal whether a landmark coordinate has an obvious relationship with pitch, yaw, or roll. This is exploratory only; the Random Forest model can use nonlinear interactions that are harder to see in two dimensions.

In [ ]:
if not features_df.empty:
    candidate_features = ["forehead_y", "left_eye_x", "right_eye_x", "chin_y", "mouth_left_x", "mouth_right_x"]
    candidate_features = [col for col in candidate_features if col in normalized_features_df.columns]

    fig, axes = plt.subplots(len(candidate_features), 3, figsize=(15, 3 * len(candidate_features)))
    if len(candidate_features) == 1:
        axes = np.array([axes])

    for row_idx, feature in enumerate(candidate_features):
        for col_idx, angle in enumerate(["pitch", "yaw", "roll"]):
            ax = axes[row_idx, col_idx]
            sns.scatterplot(
                data=normalized_features_df.sample(min(1500, len(normalized_features_df)), random_state=RANDOM_STATE),
                x=feature,
                y=angle,
                s=12,
                alpha=0.45,
                ax=ax,
            )
            ax.set_title(f"{feature} vs {angle}")
    plt.tight_layout()
    plt.show()


## Notes For Next Iteration

- If missing values are high, inspect failed images and consider whether FaceMesh confidence thresholds need tuning.
- If pose-angle distributions are heavily imbalanced, evaluate performance separately for frontal and extreme-pose subsets.
- If normalized features have extreme outliers, inspect the corresponding images for bad landmarks or annotation issues.
- Keep production extraction and training code in `src/`; use this notebook for evidence, charts, and interpretation.